# Stockage vectoriel comparé — FAISS et pgvector à côté de Qdrant

> **Complément du [05-Stockage-Vectoriel](05-Stockage-Vectoriel.ipynb)** (index HNSW à la main, compromis `ef`) et du [05b-Stockage-Vectoriel-Serveur](05b-Stockage-Vectoriel-Serveur.ipynb) (serveur Qdrant en conteneur). Ici, **trois moteurs réels** traitent le **même jeu de données**, et chaque mesure répond à **une seule question**.

**Navigation** : [Index](README.md) | [<< Précédent](05b-Stockage-Vectoriel-Serveur.ipynb)

## 1. Objectif — une question par mesure

| # | Mesure | Question pédagogique |
|---|---|---|
| 1 | **Rappel@10** | Que perd l'approximatif, et à quel réglage ? |
| 2 | **Latence** (médiane, 200 requêtes) | Que coûte une recherche, par architecture ? |
| 3 | **Empreinte mémoire** (mesurée) | Qui paie quoi : le brut, le graphe, le payload ? |
| 4 | **Filtrage métadonnées** | Qui filtre **avant** l'index ANN, qui filtre **après** ? |

Les trois moteurs :

- **FAISS** (`faiss-cpu`) — bibliothèque C++ d'index, sans service : le vecteur est roi, les métadonnées n'existent pas ;
- **Qdrant** en mode local in-process (celui du 05) — moteur avec payload et **pré-filtre intégré à l'ANN** ;
- **PostgreSQL + pgvector** — la base relationnelle avec un index HNSW : les métadonnées sont des **colonnes SQL**, le filtre est une clause `WHERE`.

Le verdict attendu n'est pas un classement de boutique : c'est **où les moteurs divergent réellement** (exact contre approché, pré-filtre contre post-filtre).

In [1]:
import json
import os
import shutil
import subprocess
import tempfile
import time

import numpy as np

# --- Le meme jeu que 05b : 10 000 vecteurs de dim 128, 50 grappes, payload semantique ---
rng = np.random.default_rng(20261018)
N_VECTORS, DIM, N_CLUSTERS = 10_000, 128, 50

centers = rng.normal(0, 1.0, (N_CLUSTERS, DIM)).astype(np.float32)
cluster_ids = rng.integers(0, N_CLUSTERS, N_VECTORS)
vectors = (centers[cluster_ids] + rng.normal(0, 0.35, (N_VECTORS, DIM))).astype(np.float32)
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)  # L2-normalises : distance L2 ~ cosinus

SOURCES = ("incident", "wiki", "manuel")
payload = [
    {"source": SOURCES[i % 3], "severite": int(1 + (i * 7) % 5), "cluster": int(cluster_ids[i])}
    for i in range(N_VECTORS)
]
from collections import Counter
print(f"Jeu de donnees : {vectors.shape} vecteurs L2-normalises, {N_CLUSTERS} grappes")
print("Repartition sources :", dict(Counter(p["source"] for p in payload)))
print("Severites :", dict(sorted(Counter(p["severite"] for p in payload).items())))

Jeu de donnees : (10000, 128) vecteurs L2-normalises, 50 grappes
Repartition sources : {'incident': 3334, 'wiki': 3333, 'manuel': 3333}
Severites : {1: 2000, 2: 2000, 3: 2000, 4: 2000, 5: 2000}


### Lecture du résultat

`vectors.shape = (10000, 128)` : dix mille vecteurs **L2-normalisés** répartis en **50 grappes** — le même générateur que le 05b, graine `20261018` comprise, pour que les chiffres des trois carnets se recoupent. Le payload est **délibérément déséquilibré** : `incident` ne couvre qu'un tiers des lignes, et croiser `source = 'incident'` avec `severite >= 3` laisse **20 %** du corpus — c'est ce filtre sélectif qui révélera la divergence pré-filtre / post-filtre de la section 6.

In [2]:
# --- La verite : k-NN exact numpy sur 200 requetes (le meme protocole que 05b) ---
N_QUERIES, K = 200, 10
queries = vectors[rng.choice(N_VECTORS, N_QUERIES, replace=False)].copy()

def exact_topk(q, k=K):
    """k-NN exact : distance a TOUTES les lignes, tri complet."""
    d = np.linalg.norm(vectors - q, axis=1)
    return set(np.argsort(d)[:k].tolist())

t0 = time.perf_counter()
ground_truth = [exact_topk(q) for q in queries]
print(f"Ground truth exact : {N_QUERIES} requetes x k={K} calculees en {time.perf_counter() - t0:.2f} s")
print("Reference : rappel 1.000 par construction — tout moteur se compare a ceci.")

Ground truth exact : 200 requetes x k=10 calculees en 1.20 s
Reference : rappel 1.000 par construction — tout moteur se compare a ceci.


### Lecture du résultat

La vérité est **calculée hors des moteurs** : `argsort` numpy sur les 10 000 distances, 200 requêtes. Tout rappel mesuré ensuite se lit comme *la fraction des vrais k plus proches voisins que le moteur retrouve*. C'est le protocole du 05b — volontairement identique, pour que les nombres soient comparables d'un carnet à l'autre.

## 2. Provisionner les trois moteurs — fail-visible

Un moteur absent **s'affiche**, il ne se maquille pas : FAISS est importé, Qdrant ouvre un store local dans un répertoire temporaire (le mode `path=` du 05), et pgvector tourne dans un **conteneur Docker** `pgvector/pgvector:pg16` sur le port 5433 — provisionné de façon idempotente. Les sections suivantes mesurent ce qui est debout ; chaque section du moteur manquant est sautée et signalée dans sa sortie, et la garde finale résume ce qui a manqué.

In [3]:
# --- Sonde des trois moteurs, avant toute mesure ---
def docker_available():
    try:
        r = subprocess.run(["docker", "info"], capture_output=True, timeout=10)
        return r.returncode == 0
    except Exception:
        return False

faiss_ok = False
try:
    import faiss
    faiss_ok = True
except ImportError as e:
    print("FAISS indisponible :", e)

qdrant_ok = False
try:
    from qdrant_client import QdrantClient, models
    qdrant_ok = True
except ImportError as e:
    print("qdrant-client indisponible :", e)

docker_ok = docker_available()
pg_container = "coursia-pgvector-05c"
pg_ok = False
if docker_ok:
    running = subprocess.run(["docker", "inspect", "-f", "{{.State.Running}}", pg_container],
                             capture_output=True, text=True,
                             encoding="utf-8", errors="replace").stdout.strip()
    if running != "true":
        subprocess.run(["docker", "rm", "-f", pg_container], capture_output=True)
        subprocess.run(["docker", "run", "-d", "--name", pg_container,
                        "-e", "POSTGRES_PASSWORD=bench05c", "-p", "127.0.0.1:5433:5432",
                        "pgvector/pgvector:pg16"], capture_output=True)
    for _ in range(30):
        r = subprocess.run(["docker", "exec", pg_container, "pg_isready", "-U", "postgres"],
                           capture_output=True)
        if r.returncode == 0:
            pg_ok = True
            break
        time.sleep(1)

print(f"FAISS import : {faiss_ok}")
print(f"Qdrant in-process : {qdrant_ok}")
print(f"Docker : {docker_ok} -> conteneur pgvector pret : {pg_ok}")

FAISS import : True
Qdrant in-process : True
Docker : True -> conteneur pgvector pret : True


In [4]:
# --- Qdrant local : collection + ingestion (le mode path= du 05) ---
qdrant_dir = tempfile.mkdtemp(prefix="qdrant_05c_")
qclient = QdrantClient(path=qdrant_dir) if qdrant_ok else None
if not qdrant_ok:
    print("Qdrant indisponible -- section sautee (mode degrade, voir garde finale)")
else:
    qclient.create_collection(
        collection_name="bench05c",
        vectors_config=models.VectorParams(size=DIM, distance=models.Distance.COSINE),
        hnsw_config=models.HnswConfigDiff(m=16, ef_construct=200),
    )
    qclient.upload_points(
        collection_name="bench05c",
        points=[models.PointStruct(id=i, vector=vectors[i].tolist(), payload=payload[i])
                for i in range(N_VECTORS)],
        wait=True,
    )
    print(f"Qdrant : collection bench05c, {qclient.count('bench05c').count} points charges")

Qdrant : collection bench05c, 10000 points charges


In [5]:
# --- pgvector : extension, table, ingestion, reference seq-scan, index HNSW ---
import psycopg

def vec_lit(v):
    return "[" + ",".join(f"{x:.6f}" for x in v) + "]"

pg = None
pg_seqscan = None
if not pg_ok:
    print("pgvector indisponible -- section sautee (mode degrade, voir garde finale)")
else:
    pg = psycopg.connect("host=127.0.0.1 port=5433 dbname=postgres user=postgres password=bench05c",
                         autocommit=True)
    cur = pg.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
    cur.execute("DROP TABLE IF EXISTS bench05c")
    cur.execute("CREATE TABLE bench05c (id integer PRIMARY KEY, source text,"
                " severite integer, cluster integer, embedding vector(128))")
    t0 = time.perf_counter()
    cur.executemany(
        "INSERT INTO bench05c VALUES (%s, %s, %s, %s, %s::vector)",
        [(i, payload[i]["source"], payload[i]["severite"], payload[i]["cluster"], vec_lit(vectors[i]))
         for i in range(N_VECTORS)],
    )
    print(f"pgvector : {N_VECTORS} lignes inserees en {time.perf_counter() - t0:.1f} s")
    # Reference exacte cote SQL : seq-scan, mesure AVANT que l'index ANN n'existe
    ts_scan, hits_scan = [], 0
    for i, q in enumerate(queries):
        t0 = time.perf_counter()
        cur.execute("SELECT id FROM bench05c ORDER BY embedding <=> %s LIMIT %s", (vec_lit(q), K))
        ids = {r[0] for r in cur.fetchall()}
        ts_scan.append(time.perf_counter() - t0)
        hits_scan += len(ids & ground_truth[i])
    pg_seqscan = {"recall": hits_scan / (N_QUERIES * K), "ms": float(np.median(ts_scan)) * 1000}
    print(f"pg seq-scan exact (sans index) : rappel@10={pg_seqscan['recall']:.3f}"
          f" med={pg_seqscan['ms']:.2f} ms")
    t0 = time.perf_counter()
    cur.execute("CREATE INDEX ON bench05c USING hnsw (embedding vector_cosine_ops)"
                " WITH (m = 16, ef_construction = 200)")
    cur.execute("ANALYZE bench05c")
    print(f"index HNSW pgvector construit en {time.perf_counter() - t0:.1f} s")

pgvector : 10000 lignes inserees en 1.9 s


pg seq-scan exact (sans index) : rappel@10=1.000 med=6.34 ms


index HNSW pgvector construit en 7.7 s


### Interprétation

Trois architectures maintenant debout sur le même jeu : FAISS tient l'index **dans la mémoire du processus** (aucun service), Qdrant a écrit sa collection **sur disque** (`path=`), et Postgres a ingéré les vecteurs **comme des colonnes d'une table** avec un index HNSW. L'ingestion pgvector est la plus verbeuse — chaque vecteur devient une ligne SQL — mais elle achète quelque chose que FAISS n'a pas : les métadonnées vivent **dans le même stockage**, interrogeables par `WHERE`. Et le seq-scan exact mesuré ici (quelques millisecondes sur 10k lignes) sert de point d'entrée à la section latence : c'est ce que l'index ANN viendra remplacer.

## 3. Rappel@10 — ce que l'approximatif perd, et à quel réglage

Chaque moteur cherche les k = 10 plus proches voisins des 200 requêtes ; le rappel est la fraction des vrais voisins retrouvés. Le paramètre d'exploration — `efSearch` chez FAISS, `hnsw_ef` chez Qdrant, `hnsw.ef_search` chez pgvector — est **le même concept** : la taille de la liste de candidats pendant la traversée du graphe.

In [6]:
# --- FAISS : Flat (exact, la reference interne) puis HNSW a efSearch variable ---
def recall_faiss(index, k=K):
    _, I = index.search(queries, k)
    return float(np.mean([len(set(I[i].tolist()) & ground_truth[i]) / k for i in range(N_QUERIES)]))

recall_rows = []
faiss_flat, faiss_hnsw = None, None
if not faiss_ok:
    print("FAISS indisponible -- section sautee (mode degrade, voir garde finale)")
else:
    faiss_flat = faiss.IndexFlatL2(DIM)
    faiss_flat.add(vectors)
    recall_rows.append(("FAISS Flat (exact)", "-", recall_faiss(faiss_flat)))
    faiss_hnsw = faiss.IndexHNSWFlat(DIM, 16)  # M = 16, comme Qdrant et pgvector
    faiss_hnsw.hnsw.efConstruction = 200
    faiss_hnsw.add(vectors)
    for ef in (8, 16, 32, 64):
        faiss_hnsw.hnsw.efSearch = ef
        recall_rows.append(("FAISS HNSW", f"efSearch={ef}", recall_faiss(faiss_hnsw)))
    faiss_hnsw.hnsw.efSearch = 64  # reglage par defaut pour la suite

for name, ef, r in recall_rows:
    print(f"{name:<18s} {ef:<12s} rappel@10 = {r:.3f}")

FAISS Flat (exact) -            rappel@10 = 1.000
FAISS HNSW         efSearch=8   rappel@10 = 0.965
FAISS HNSW         efSearch=16  rappel@10 = 0.993
FAISS HNSW         efSearch=32  rappel@10 = 0.998
FAISS HNSW         efSearch=64  rappel@10 = 1.000


In [7]:
# --- Qdrant puis pgvector : le meme balayage du parametre d'exploration ---
def recall_qdrant(ef, k=K):
    hits = 0
    for i, q in enumerate(queries):
        res = qclient.query_points("bench05c", query=q.tolist(), limit=k,
                                   search_params=models.SearchParams(hnsw_ef=ef, exact=False)).points
        hits += len({p.id for p in res} & ground_truth[i])
    return hits / (N_QUERIES * k)

if not qdrant_ok:
    print("Qdrant indisponible -- balayage saute (mode degrade)")
else:
    for ef in (8, 16, 32, 64):
        print(f"Qdrant in-proc       hnsw_ef={ef:<6d} rappel@10 = {recall_qdrant(ef):.3f}")

def recall_pg(ef, k=K):
    cur.execute(f"SET hnsw.ef_search = {ef}")
    hits = 0
    for i, q in enumerate(queries):
        cur.execute("SELECT id FROM bench05c ORDER BY embedding <=> %s LIMIT %s", (vec_lit(q), k))
        hits += len({r[0] for r in cur.fetchall()} & ground_truth[i])
    return hits / (N_QUERIES * k)

if not pg_ok:
    print("pgvector indisponible -- balayage saute (mode degrade)")
else:
    for ef in (4, 8, 16, 40):
        print(f"pgvector HNSW        ef_search={ef:<4d} rappel@10 = {recall_pg(ef):.3f}")

Qdrant in-proc       hnsw_ef=8      rappel@10 = 1.000


Qdrant in-proc       hnsw_ef=16     rappel@10 = 1.000


Qdrant in-proc       hnsw_ef=32     rappel@10 = 1.000


Qdrant in-proc       hnsw_ef=64     rappel@10 = 1.000


pgvector HNSW        ef_search=4    rappel@10 = 0.400


pgvector HNSW        ef_search=8    rappel@10 = 0.800


pgvector HNSW        ef_search=16   rappel@10 = 1.000


pgvector HNSW        ef_search=40   rappel@10 = 1.000


### Lecture du résultat

Le flat est exact : rappel 1.000, par construction. La comparaison tient en une colonne : à `ef = 8` pour k = 10, **le même réglage ne produit pas le même rappel** — FAISS rend 0.965, Qdrant 1.000, et pgvector s'effondre (0.400 à `ef_search = 4`, 0.800 à 8) : chez FAISS et pgvector, une liste de candidats plus courte que k perd mécaniquement des voisins réels ; l'implémentation Qdrant, elle, tient le coup ici. Même nom de paramètre, **sémantiques internes différentes**. Dès que `ef` dépasse k, tous remontent à 1.000 : à cette échelle (10k vecteurs, grappes), le compromis vit **dans les réglages bas**, exactement comme la courbe signature du 05b l'avait montré sur le serveur Qdrant. Ce n'est pas un défaut : c'est **le prix de la vitesse**, que la section suivante mesure.

## 4. Latence — médiane sur 200 requêtes

Un single-shot ment (cache chaud, GC, Turbo) ; la **médiane sur 200 requêtes** tient. Chaque moteur cherche k = 10 avec son réglage nominal (`ef` = 64 FAISS/Qdrant, `ef_search` = 40 pgvector), plus deux références mesurées : le **flat exact FAISS** et le **seq-scan SQL** de Postgres — le `ORDER BY ... LIMIT` sans index ANN, mesuré au provisionnement avant la création de l'index.

In [8]:
# --- Medianes par moteur, k=10, reglages nominaux ---
def med_lat(fn):
    ts = []
    for q in queries:
        t0 = time.perf_counter()
        fn(q)
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts)) * 1000

lat_rows = []
if faiss_ok:
    lat_rows += [
        ("FAISS Flat (exact)", med_lat(lambda q: faiss_flat.search(q.reshape(1, -1), K))),
        ("FAISS HNSW ef=64", med_lat(lambda q: faiss_hnsw.search(q.reshape(1, -1), K))),
    ]
if qdrant_ok:
    lat_rows.append(("Qdrant in-proc ef=64", med_lat(
        lambda q: qclient.query_points("bench05c", query=q.tolist(), limit=K,
                                       search_params=models.SearchParams(hnsw_ef=64)))))
if pg_ok:
    lat_rows.append(("pg seq-scan (exact, sans index)", pg_seqscan["ms"]))
    cur.execute("SET hnsw.ef_search = 40")
    lat_rows.append(("pgvector HNSW ef=40", med_lat(
        lambda q: cur.execute("SELECT id FROM bench05c ORDER BY embedding <=> %s LIMIT %s",
                              (vec_lit(q), K)) or cur.fetchall())))

print(f"{'moteur':<30s} {'mediane (ms)':>12s}")
for name, ms in lat_rows:
    print(f"{name:<30s} {ms:>12.3f}")

moteur                         mediane (ms)
FAISS Flat (exact)                    0.328
FAISS HNSW ef=64                      0.058
Qdrant in-proc ef=64                 12.630
pg seq-scan (exact, sans index)        6.337
pgvector HNSW ef=40                   1.605


### Lecture du résultat

Un facteur **deux cents** entre les extrêmes, trois architectures et non trois qualités. FAISS natif répond en centièmes de milliseconde (0.058 ms) — la bibliothèque C++ ne traverse aucune couche. pgvector paie l'aller-retour SQL (1.6 ms), Qdrant in-process paie son client Python (12.6 ms). Détail qui enseigne : sur ce jouet de 10k vecteurs, le client Qdrant in-process coûte **le double** du seq-scan SQL complet (12.6 contre 6.3 ms) — le traverser en Python peut coûter plus cher que tout relire en SQL ; personne n'est pénalisé à cette échelle, tout le monde le sera quand le volume grandit (millions de lignes, le mur de latence du 05) : le seq-scan croît linéairement, l'index ANN reste plat. La leçon n'est pas « FAISS gagne » : le choix d'architecture se fait sur **les sections 5 et 6**, pas sur ces millièmes de seconde.

## 5. Empreinte mémoire — mesurée, pas déclarée

Trois façons de mesurer un index : FAISS se **sérialise** (octets exacts du blob), Qdrant écrit sa collection **sur disque** (taille du répertoire), Postgres expose `pg_total_relation_size` (table + index). Ce ne sont pas les mêmes objets — c'est précisément ce que la table montre.

In [9]:
# --- Empreintes mesurees par moteur ---
mem_rows = []
if faiss_ok:
    mem_rows += [
        ("FAISS Flat", faiss.serialize_index(faiss_flat).nbytes / 1024),
        ("FAISS HNSW (M=16)", faiss.serialize_index(faiss_hnsw).nbytes / 1024),
    ]
if qdrant_ok:
    disk = sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(qdrant_dir) for f in fs)
    mem_rows.append(("Qdrant collection (disque)", disk / 1024))
if pg_ok:
    cur.execute("SELECT pg_relation_size('bench05c_embedding_idx')")
    mem_rows.append(("pgvector index HNSW", cur.fetchone()[0] / 1024))
    cur.execute("SELECT pg_total_relation_size('bench05c')")
    mem_rows.append(("pgvector table+index total", cur.fetchone()[0] / 1024))

brut = vectors.nbytes / 1024
print(f"{'objet':<28s} {'taille (Ko)':>11s}")
print(f"{'brut numpy (reference)':<28s} {brut:>11.0f}")
for name, ko in mem_rows:
    print(f"{name:<28s} {ko:>11.0f}")

objet                        taille (Ko)
brut numpy (reference)              5000
FAISS Flat                          5000
FAISS HNSW (M=16)                   6407
Qdrant collection (disque)         20357
pgvector index HNSW                 8160
pgvector table+index total         14160


### Lecture du résultat

Le **brut** (10k × 128 × float32 = 5 000 Ko) est la référence. Le flat FAISS le contient exactement — l'exact est ici le plus léger. Le graphe HNSW ajoute ses liens : environ +30 % chez FAISS ; l'index pgvector dépasse le brut d'une bonne moitié (liens **et** pointeurs d'articles — la structure interne n'est pas la même, pour le même M = 16). Qdrant paie le plus lourd : collection sur disque, **payload stocké et segments internes** — la contrepartie de l'interroger en filtre à la section suivante. Aucune de ces tailles n'est « la mémoire de l'ANN » en général : chacune mesure **ce que ce moteur choisit de matérialiser**.

## 6. Filtrage métadonnées — pré-filtre contre post-filtre, la vraie divergence

Même requête pour tous : les k = 10 plus proches voisins **parmi les lignes** `source = 'incident' AND severite >= 3` (20 % du corpus, mesuré). Les moteurs ne répondent pas avec le même mécanisme :

- **Qdrant** : le filtre entre dans la traversée HNSW — l'ANN ne visite que les points qui passent le filtre (**pré-filtre intégré**) ;
- **pgvector** : la clause `WHERE` s'applique au flux de candidats pendant le scan — le filtrage vit **dans le moteur SQL, au ras de l'index** ;
- **FAISS** : l'index ne connaît pas les métadonnées. Il faut chercher **plus** de voisins puis filtrer (**post-filtre**) — combien élargir, quand le filtre ne laisse que 20 % ?

In [10]:
# --- Le meme filtre selectif, trois mecanismes ---
FILTER = lambda i: payload[i]["source"] == "incident" and payload[i]["severite"] >= 3
allowed = {i for i in range(N_VECTORS) if FILTER(i)}
print(f"Filtre : source='incident' AND severite>=3 -> {len(allowed)} lignes admises"
      f" ({100 * len(allowed) / N_VECTORS:.0f}% du corpus)")

if qdrant_ok:
    # Qdrant : pre-filtre integre a la traversee
    res_f = qclient.query_points(
        "bench05c", query=queries[0].tolist(), limit=K,
        query_filter=models.Filter(must=[
            models.FieldCondition(key="source", match=models.MatchValue(value="incident")),
            models.FieldCondition(key="severite", range=models.Range(gte=3)),
        ]),
        search_params=models.SearchParams(hnsw_ef=64),
    ).points
    print(f"Qdrant pre-filtre   : {[p.id for p in res_f]}")

if pg_ok:
    # pgvector : WHERE dans le SQL
    cur.execute("SET hnsw.ef_search = 40")
    cur.execute("SELECT id FROM bench05c WHERE source = 'incident' AND severite >= 3"
                " ORDER BY embedding <=> %s LIMIT %s", (vec_lit(queries[0]), K))
    print(f"pgvector WHERE      : {[r[0] for r in cur.fetchall()]}")

if faiss_ok:
    # FAISS : post-filtre -- il faut demander PLUS que k pour reussir a en rendre k
    for probe in (K, 5 * K, 10 * K):
        _, I = faiss_hnsw.search(queries[0].reshape(1, -1), probe)
        kept = [int(i) for i in I[0] if FILTER(int(i))][:K]
        print(f"FAISS post-filtre k*={probe:<4d}: {kept}"
              f"{'  <-- pas assez elargi' if len(kept) < K else ''}")

Filtre : source='incident' AND severite>=3 -> 2000 lignes admises (20% du corpus)


Qdrant pre-filtre   : [5667, 6807, 4782, 8826, 1881, 7002, 4257, 762, 9462, 3216]
pgvector WHERE      : [5667, 6807, 4782, 8826]
FAISS post-filtre k*=10  : []  <-- pas assez elargi
FAISS post-filtre k*=50  : [5667, 6807, 4782, 8826, 1881, 7002, 4257]  <-- pas assez elargi
FAISS post-filtre k*=100 : [5667, 6807, 4782, 8826, 1881, 7002, 4257, 762, 9462, 3216]


### Lecture du résultat

Trois comportements mesurés, aucun ne se ressemble. **Qdrant rend dix admis d'emblée** : le filtre fait partie de la traversée. **pgvector n'en rend que quatre** ici — avec les réglages par défaut, le scan HNSW filtré s'arrête sans garantir d'atteindre `LIMIT` ; la documentation pgvector nomme le remède (`hnsw.iterative_scan`), désactivé par défaut — à activer (ou `ef_search` à élargir) si vos filtres sont sélectifs. **FAISS rend zéro, puis sept, puis dix** : à k* = 10 aucun candidat admis sur cette requête (statistiquement ~2 attendus à 20 % d'admis), à k* = 50 il en manque encore, à k* = 100 la liste est enfin pleine. Le post-filtre **fonctionne, mais il paie** : il faut sur-consommer des candidats, et le rappel mesuré au sol (contre le ground truth filtré — l'exercice 2 le mesure) dégrade quand le filtre devient plus sélectif. C'est LA divergence réelle entre moteurs — celle qui coûte en production quand le filtre est sélectif.

## 7. Verdict — ce que chaque moteur enseigne

| Moteur | Ce qu'il enseigne | Ce qu'il ne fait pas |
|---|---|---|
| **FAISS** | le compromis exact/approché dans sa forme la plus pure, l'empreinte minimale, la vitesse native | ne connaît ni métadonnées ni filtre : post-filtre à votre charge |
| **Qdrant** | le pré-filtre **dans** l'ANN, le payload comme citoyen de première classe | (in-process) une latence par requête dominée par le client Python |
| **pgvector** | les vecteurs **dans** la base relationnelle : un `WHERE` filtrant, transactionnel, avec vos jointures | la latence SQL par requête ; l'index ANN se construit et se mesure comme un index B-tree |

Le classement de boutique n'existe pas : à 10k vecteurs tout est confortable, et c'est **la question** qui choisit le moteur. Filtrer en SQL au milieu de jointures métier → pgvector ; comparer des index purs à très grande échelle → FAISS ; servir du RAG filtré sans opérer un SGBD → Qdrant. Le 05 vous a montré le HNSW **à la main**, le 05b **en serveur** : ce carnet a fermé la boucle en montrant que les trois moteurs implémentent le même compromis — et divergent sur le filtrage.

## 8. Exercices et exemples guidés

Cette section mêle deux types de cellules — à ne pas confondre.

**Trois exemples guidés** (cellules marquées *Exemple guidé*) sont **résolus** : leurs sorties sont celles du code committé, et la difficulté est de **lire** la sortie.
**Trois exercices** (cellules *Exercice*) sont **à compléter** : le squelette, les indices et les `# TODO étudiant` sont fournis ; le carnet s'exécute de bout en bout même non complété (les fonctions retournent `None` et les sorties le disent).

In [11]:
# Exemple guidé 1 (RESOLU) : le compromis rappel/ef de FAISS quand k grandit.
# Meme protocole que la section 3, mais on balaye k a ef fixe -- la liste de candidats
# doit depasser k, sinon le rappel plafonne mecaniquement.
def compromis_k(k_list=(5, 10, 20), ef_list=(8, 16, 32)):
    rows = []
    for ef in ef_list:
        faiss_hnsw.hnsw.efSearch = ef
        for k in k_list:
            _, I = faiss_hnsw.search(queries, k)
            gt_k = [set(np.argsort(np.linalg.norm(vectors - q, axis=1))[:k].tolist()) for q in queries]
            r = float(np.mean([len(set(I[i].tolist()) & gt_k[i]) / k for i in range(N_QUERIES)]))
            rows.append((ef, k, r))
    faiss_hnsw.hnsw.efSearch = 64
    return rows

if not faiss_ok:
    print("FAISS indisponible -- exemple guidé 1 saute (mode degrade)")
else:
    for ef, k, r in compromis_k():
        print(f"ef={ef:<3d} k={k:<3d} rappel@k = {r:.3f}")

ef=8   k=5   rappel@k = 0.984
ef=8   k=10  rappel@k = 0.965
ef=8   k=20  rappel@k = 0.915
ef=16  k=5   rappel@k = 0.998
ef=16  k=10  rappel@k = 0.993
ef=16  k=20  rappel@k = 0.980
ef=32  k=5   rappel@k = 1.000
ef=32  k=10  rappel@k = 0.998
ef=32  k=20  rappel@k = 0.996


### Exercice 1 — le compromis côté SQL

L'exemple guidé 1 a fait apparaître le plafond `ef < k` côté FAISS. **pgvector a le même réglage** (`hnsw.ef_search`). Faites apparaître le compromis côté SQL : balayez `ef_search` sous la valeur k et mesurez le rappel — la documentation pgvector dit que le scan itératif élargit de lui-même, la question est : **ce rattrapage se voit-il dans le rappel mesuré ?**

In [12]:
# Exercice 1 (a completer) : rappel pgvector quand ef_search descend sous k.
def compromis_pg(ef_list=(4, 8, 12), k=K):
    # TODO Etudiant :
    # (1) pour chaque ef de ef_list : SET hnsw.ef_search = ef
    # (2) mesurer le rappel@k contre ground_truth (meme boucle que recall_pg)
    # (3) retourner la liste [(ef, rappel), ...]
    print("Exercice a completer : compromis_pg")
    return None

resultat_exo1 = compromis_pg()
if resultat_exo1 is not None:
    for ef, r in resultat_exo1:
        print(f"ef_search={ef:<3d} rappel@10 = {r:.3f}")

Exercice a completer : compromis_pg


In [13]:
# Exemple guidé 2 (RESOLU) : le filtre qui ne matche rien -- le comportement vide compare.
# Qdrant : le pre-filtre integre rend une liste vide, immediatement.
# FAISS : le post-filtre rend vide SEULEMENT apres avoir sur-consomme k* candidats.
if qdrant_ok:
    filtre_impossible = models.Filter(must=[
        models.FieldCondition(key="source", match=models.MatchValue(value="incident")),
        models.FieldCondition(key="source", match=models.MatchValue(value="wiki")),  # contradictoire
    ])
    res_vide = qclient.query_points("bench05c", query=queries[0].tolist(), limit=K,
                                    query_filter=filtre_impossible,
                                    search_params=models.SearchParams(hnsw_ef=64)).points
    print(f"Qdrant filtre contradictoire : {len(res_vide)} resultat (immediat)")

if faiss_ok:
    _, I_vide = faiss_hnsw.search(queries[0].reshape(1, -1), 100)
    kept_vide = [int(i) for i in I_vide[0]
                 if payload[int(i)]["source"] == "incident" and payload[int(i)]["source"] == "wiki"]
    print(f"FAISS post-filtre (meme condition) : {len(kept_vide)} resultat apres avoir"
          f" parcouru {len(I_vide[0])} candidats")

Qdrant filtre contradictoire : 0 resultat (immediat)
FAISS post-filtre (meme condition) : 0 resultat apres avoir parcouru 100 candidats


### Exercice 2 — le prix réel du post-filtre FAISS

L'exemple guidé 2 a montré le cas vide. Le cas intéressant en production : un filtre **sélectif mais pas vide**. Post-filtrez FAISS avec `severite >= 4` (40 % des lignes admises ici) en élargissant k* = 10, 30, 100, et mesurez le **rappel contre le ground truth filtré** (les vrais k plus proches voisins *parmi les lignes admises*) : combien de sur-consommation faut-il pour ne rien perdre ?

In [14]:
# Exercice 2 (a completer) : rappel du post-filtre FAISS contre ground truth FILTRE.
def postfiltre_faiss(seuil=4, k_list=(10, 30, 100), k=K):
    # TODO Etudiant :
    # (1) construire allowed = {i : payload[i]['severite'] >= seuil}
    # (2) ground truth filtre : exact_topk restreint a allowed (parmi les admis)
    # (3) pour chaque k* : search k*, garder les admis, trancher a k, mesurer le rappel
    # (4) retourner [(k*, rappel), ...]
    print("Exercice a completer : postfiltre_faiss")
    return None

resultat_exo2 = postfiltre_faiss()
if resultat_exo2 is not None:
    for kstar, r in resultat_exo2:
        print(f"k*={kstar:<4d} rappel (vs GT filtre) = {r:.3f}")

Exercice a completer : postfiltre_faiss


In [15]:
# Exemple guidé 3 (RESOLU) : la persistance pgvector -- le conteneur redemarre, pas les donnees.
# Miroir exact du test de redemarrage du 05b (serveur Qdrant), transpose au SGBD.
if not pg_ok:
    print("pgvector indisponible -- exemple guidé 3 saute (mode degrade)")
else:
    cur.execute("SELECT COUNT(*) FROM bench05c")
    avant = cur.fetchone()[0]
    subprocess.run(["docker", "restart", pg_container], capture_output=True, check=True)
    for _ in range(30):
        if subprocess.run(["docker", "exec", pg_container, "pg_isready", "-U", "postgres"],
                          capture_output=True).returncode == 0:
            break
        time.sleep(1)
    pg2 = psycopg.connect("host=127.0.0.1 port=5433 dbname=postgres user=postgres password=bench05c",
                          autocommit=True)
    cur2 = pg2.cursor()
    cur2.execute("SELECT COUNT(*) FROM bench05c")
    apres = cur2.fetchone()[0]
    cur2.execute("SELECT id FROM bench05c ORDER BY embedding <=> %s LIMIT %s", (vec_lit(queries[0]), K))
    ids_apres = [r[0] for r in cur2.fetchall()]
    print(f"count avant redemarrage : {avant} / apres : {apres}")
    print(f"recherche k=10 apres redemarrage : {ids_apres}")
    print("L'index HNSW et la table survivent au restart : la collection est DANS le volume du conteneur.")

count avant redemarrage : 10000 / apres : 10000
recherche k=10 apres redemarrage : [3408, 3427, 5880, 7786, 1366, 5089, 1638, 9514, 5741, 4413]
L'index HNSW et la table survivent au restart : la collection est DANS le volume du conteneur.


### Exercice 3 — la mémoire de l'IVF

La section 5 a mesuré flat et HNSW. FAISS a une **troisième famille** : les index IVF (partitionnement en cellules, on sonde `nprobe` cellules). Construisez un `IndexIVFFlat` (nlist = 50, comme les grappes), mesurez son empreinte sérialisée et son rappel à `nprobe` variable — l'empreinte se rapproche-t-elle du flat, et le compromis `nprobe` ressemble-t-il au compromis `ef` ?

In [16]:
# Exercice 3 (a completer) : empreinte et rappel d'un index IVF.
def ivf_mesures(nlist=50, nprobe_list=(1, 5, 20), k=K):
    # TODO Etudiant :
    # (1) quantizer = faiss.IndexFlatL2(DIM) ; ivf = faiss.IndexIVFFlat(quantizer, DIM, nlist)
    # (2) ivf.train(vectors) ; ivf.add(vectors)  # train AVANT add (IVF apprend ses cellules)
    # (3) taille : faiss.serialize_index(ivf).nbytes / 1024
    # (4) pour chaque nprobe : ivf.nprobe = nprobe, mesurer le rappel comme en section 3
    # (5) retourner (taille_ko, [(nprobe, rappel), ...])
    print("Exercice a completer : ivf_mesures")
    return None

resultat_exo3 = ivf_mesures()
if resultat_exo3 is not None:
    taille, rows = resultat_exo3
    print(f"empreinte IVF : {taille:.0f} Ko")
    for nprobe, r in rows:
        print(f"nprobe={nprobe:<3d} rappel@10 = {r:.3f}")

Exercice a completer : ivf_mesures


## Garde finale — ce qui a tourné, ce qui a manqué

Le carnet s'exécute de bout en bout **quoi qu'il arrive** : chaque moteur absent affiche sa section sautée et les mesures restantes se font avec ceux qui sont debout. La cellule suivante rappelle lequel des trois a manqué sur cette machine, s'il y en a eu un.

In [17]:
# Garde finale : rappel de disponibilite mesuree en debut de carnet.
print("Moteurs debout au debut du carnet :")
print(f"  FAISS             : {faiss_ok}")
print(f"  Qdrant in-proc    : {qdrant_ok}")
print(f"  pgvector (Docker) : {pg_ok}")
if not (faiss_ok and qdrant_ok and pg_ok):
    print("Mode degrade : les sections du moteur manquant sont sautees et signalees"
          " dans leurs sorties -- installer le moteur puis re-executer pour les"
          " mesures completes.")

Moteurs debout au debut du carnet :
  FAISS             : True
  Qdrant in-proc    : True
  pgvector (Docker) : True


## Conclusion

Ce carnet a mis **trois moteurs réels** devant le **même jeu** et la même vérité exacte, et chaque mesure a répondu à une question :

1. **Rappel** : le compromis vit dans les réglages bas du paramètre d'exploration — le même concept (`efSearch` / `hnsw_ef` / `ef_search`) dans les trois moteurs ; le plafond mécanique quand `ef < k` s'est vu chez FAISS et pgvector, pas chez Qdrant : même nom, sémantiques internes différentes ;
2. **Latence** : un facteur deux cents entre natif et client in-process (0.06 → 12.6 ms) = trois architectures (bibliothèque native, client in-process, SQL) — confortable à cette échelle, décisif quand le seq-scan explose ;
3. **Mémoire** : le brut n'est jamais ce qu'on paie — le flat le contient, HNSW ajoute ses liens, Qdrant matérialise payload et segments ;
4. **Filtrage** : la vraie divergence — Qdrant pré-filtre **dans** la traversée et rend k résultats, pgvector filtre dans le moteur SQL mais peut rendre **moins** que k tant que le scan itératif n'est pas activé, FAISS vous laisse le post-filtre et sa sur-consommation ;

Le [06-KernelMemory-InProcess](06-KernelMemory-InProcess.ipynb) referme la partie stockage en montrant l'orchestration au-dessus de ces briques.

In [18]:
# Nettoyage : fermer le store local et couper le client pg ; le conteneur Docker reste
# operationnel jusqu'a la fin de la seance (l'exemple guidé 3 l'a redemarre proprement).
if qdrant_ok:
    qclient.close()
    shutil.rmtree(qdrant_dir, ignore_errors=True)
if pg_ok:
    pg.close()
print("Stores locaux fermes et nettoyes. Conteneur pgvector :", pg_container,
      "(docker rm -f", pg_container, "pour le retirer).")

Stores locaux fermes et nettoyes. Conteneur pgvector : coursia-pgvector-05c (docker rm -f coursia-pgvector-05c pour le retirer).
